# Pandas API on Spark

## What's covered

- Why `pyspark.pandas` exists — the bridge between pandas ergonomics and Spark scale
- Three DataFrame types — `pandas.DataFrame`, `pyspark.pandas.DataFrame`, `pyspark.sql.DataFrame`
- Creating a `ps.DataFrame` — from a dict, from files, from a Spark DataFrame
- **Transformations** (lazy, return another `ps.DataFrame`) — column ops, filtering, `groupby`, `merge`, `apply`, accessors
- **Actions** (eager, trigger a Spark job) — `head`, `to_pandas`, frame-wide reductions, writes
- The Spark bridge — `to_spark()` / `pandas_api()` round trips
- Default index types — `sequence`, `distributed-sequence`, `distributed`
- Gotchas — `inplace=True`, row order, `ops_on_diff_frames`, and where the pandas API ends


## Why this API exists

Pandas is the lingua franca of single-machine data work — but it caps out at one machine's memory. Rewriting a pandas pipeline into the Spark DataFrame API is mechanical but tedious, and the resulting code looks nothing like the original.

`pyspark.pandas` (formerly the **Koalas** project, merged into PySpark in 3.2) gives you the pandas API on top of a Spark backend. The same `psdf.groupby(...).mean()` that ran on a laptop now runs on a cluster — distributed, lazy, and through Catalyst.

Two facts to anchor on before any code:

- **The surface is pandas. The engine is Spark.** Every `ps.DataFrame` is a thin wrapper around a `pyspark.sql.DataFrame` plus an index. Operations compile to the same Spark plans you'd write by hand.
- **The transformation / action split from notebook 01 still applies.** Even though the code reads like eager pandas, most ops are lazy. A line that *looks* like a result — `psdf['new_col'] = ...` — does no work until an action fires.

Import convention is `import pyspark.pandas as ps` — `ps` mirrors pandas' `pd`.


## Setup

We reuse the same bank data from earlier notebooks — `card_transactions` (Parquet) and `customers` (CSV). Both load as Spark DataFrames; we'll convert as needed throughout the notebook.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType,
)

spark = (
    SparkSession.builder
    .appName("PandasAPIOnSpark")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "data"

customers_schema = StructType([
    StructField("customer_id",   StringType(), nullable=False),
    StructField("full_name",     StringType()),
    StructField("email",         StringType()),
    StructField("phone",         StringType()),
    StructField("date_of_birth", DateType()),
    StructField("gender",        StringType()),
    StructField("city",          StringType()),
    StructField("state",         StringType()),
    StructField("country",       StringType()),
    StructField("created_at",    TimestampType()),
])

customers_sdf         = spark.read.schema(customers_schema).option("header", "true").csv(f"{DATA_DIR}/customers")
card_transactions_sdf = spark.read.parquet(f"{DATA_DIR}/card_transactions")

print("customers         :", customers_sdf.count(), "rows")
print("card_transactions :", card_transactions_sdf.count(), "rows")


## Three DataFrame types — keep them straight

| Type | Class | Where data lives | When to use |
|---|---|---|---|
| pandas DataFrame | `pandas.DataFrame` | Driver memory only | Final-mile analysis, plotting, small fixtures |
| Pandas-on-Spark DataFrame | `pyspark.pandas.DataFrame` | Distributed across executors | When you want pandas syntax but Spark scale |
| Spark DataFrame | `pyspark.sql.DataFrame` | Distributed across executors | Native Spark API, best performance, most features |

Only the first lives on the driver. The other two are distributed — same engine, different surface. Pandas-on-Spark adds a pandas-style index on top of a Spark DataFrame.


## Creating a `ps.DataFrame`

Three everyday entry points:

- **From a Python dict** — same constructor signature as pandas. Good for tiny fixtures.
- **From files** — `ps.read_csv`, `ps.read_parquet` go through Spark's readers under the hood.
- **From an existing Spark DataFrame** — `sdf.pandas_api()` wraps it as a `ps.DataFrame`. This is the cheap, no-data-movement path you'll use most.

The reverse direction — `pandas.DataFrame` → `ps.DataFrame` — is `ps.from_pandas(pdf)`. Use it only for fixtures; it serializes the pandas frame and re-reads it through Spark.


In [ ]:
import pyspark.pandas as ps
import pandas as pd

# 1. From a dict
psdf_dict = ps.DataFrame({
    "city":   ["Austin", "Dallas", "Seattle"],
    "score":  [85, 72, 91],
})
print("from dict :", type(psdf_dict).__module__, "shape =", psdf_dict.shape)

# 2. From files via Spark's readers
psdf_files = ps.read_parquet(f"{DATA_DIR}/card_transactions")
print("from parquet :", type(psdf_files).__module__, "shape =", psdf_files.shape)

# 3. From an existing Spark DataFrame (cheap — just a wrapper)
psdf = card_transactions_sdf.pandas_api()
ps_customers = customers_sdf.pandas_api()
print("from spark df :", type(psdf).__module__, "shape =", psdf.shape)

# Reverse: pandas → pandas-on-Spark
tiny_pdf = pd.DataFrame({"category": ["Food", "Travel"], "weight": [0.3, 0.7]})
psdf_from_pd = ps.from_pandas(tiny_pdf)
print("from pandas :", type(psdf_from_pd).__module__)


## Transformations — lazy, return another `ps.DataFrame`

Everything in this section returns a new pandas-on-Spark frame **without triggering a Spark job**. The operations build up a plan, just like Spark DataFrame transformations. An action — covered in the next section — is what fires the job.

The full toolkit divides into six families:

- **Column ops** — selection, assignment, derived columns
- **Filtering** — boolean masks, `query()`
- **Aggregation** — `groupby` + `agg`
- **Joins** — `merge`
- **Cleanup** — `fillna`, `dropna`, `astype`, `rename`, `sort_values`
- **Per-element work** — `.str` and `.dt` accessors, `apply` / `transform` with type hints


### Column ops

Pandas-style column selection, assignment, and derivation all work. Each returns a new `ps.DataFrame`; the variable being reassigned is just a new wrapper around an updated Spark plan.


In [ ]:
# Single-column projection — returns a Series
amounts = psdf["amount"]

# Multi-column projection
two_cols = psdf[["transaction_id", "amount"]]

# Derived columns via direct assignment
psdf["amount_usd"] = psdf["amount"] / 83          # fake conversion rate
psdf["amount_log"] = psdf["amount"] + 1           # placeholder for log(amount+1)

# Or the immutable assign() — returns a new frame, leaves psdf alone
psdf_with_fee = psdf.assign(fee=psdf["amount"] * 0.02)

# Nothing above has triggered a job yet — these are all lazy
print("derived columns added (no job run):", list(psdf.columns))


### Filtering

Boolean indexing and `query()` both work. Combine boolean expressions with `&`, `|`, `~` and **parenthesize each clause** — same Python operator-precedence trap as the Spark DataFrame API.


In [ ]:
# Boolean mask
big_approved = psdf[(psdf["status"] == "APPROVED") & (psdf["amount"] > 10000)]

# query() — string-based, like pandas
travel = psdf.query("merchant_category == 'Travel' and amount > 5000")

# Neither call has triggered a job — both are lazy
print(type(big_approved).__name__, "is still a", type(big_approved).__module__, "object")


### `groupby` + `agg`

The pandas idiom maps directly to a Spark `groupBy().agg(...)` under the hood. Named aggregations (the `(column, function)` tuple form) compile to the same plan you'd write with `pyspark.sql.functions`.


In [ ]:
# Average amount and transaction count by merchant category
by_category = (
    psdf.groupby("merchant_category")
        .agg(avg_amount=("amount", "mean"), n=("transaction_id", "count"))
        .sort_values("n", ascending=False)
)

# Multi-key group: (category, status)
by_cat_status = (
    psdf.groupby(["merchant_category", "status"])
        .agg(total=("amount", "sum"))
)

# Still no job — the head() call below is what fires it
print("plan built; no job yet")


### `merge` — joins by another name

`psdf1.merge(psdf2, on=..., how=...)` is the pandas spelling of a Spark join. `how` accepts `inner`, `left`, `right`, `outer` (no `left_semi` / `left_anti` — those are Spark-DataFrame-only).


In [ ]:
# Per-customer spend joined back to the customer demographics
spend_by_customer = (
    psdf[psdf["status"] == "APPROVED"]
        .groupby("customer_id")
        .agg(total_spend=("amount", "sum"), n_txns=("transaction_id", "count"))
        .reset_index()
)

enriched = spend_by_customer.merge(ps_customers, on="customer_id", how="left")
print("merged columns:", list(enriched.columns))


### Cleanup — `fillna`, `dropna`, `astype`, `rename`, `sort_values`

Same names as pandas, same semantics. All lazy.


In [ ]:
cleaned = (
    psdf
        .rename(columns={"merchant_category": "category"})
        .astype({"amount": "float64"})
        .fillna({"category": "UNKNOWN"})
        .sort_values("amount", ascending=False)
)
print("cleaned columns:", list(cleaned.columns))


### `.str` and `.dt` accessors

Vectorized string and datetime operations, identical to pandas. Internally they compile to the same JVM-resident functions used by the Spark DataFrame API — so they're fast.


In [ ]:
# .str — string ops on the merchant_name column
psdf["merchant_upper"] = psdf["merchant_name"].str.upper()
psdf["merchant_len"]   = psdf["merchant_name"].str.len()

# .dt — extract parts of transaction_at
psdf["txn_year"]  = psdf["transaction_at"].dt.year
psdf["txn_month"] = psdf["transaction_at"].dt.month
psdf["txn_dow"]   = psdf["transaction_at"].dt.dayofweek

print("derived str/dt columns:", [c for c in psdf.columns if c.startswith(('merchant_', 'txn_'))])


### `apply` / `transform` — the per-row escape hatch

When the built-in vectorized operations don't cover what you need, you can apply a Python function row-by-row. **Always pass a type hint** — without it, pandas-on-Spark has to run the function on the first few rows to infer the return type, which is slow and brittle on large data.

The type hint goes on the function signature (Python's standard return-type annotation). Spark uses it to build the schema of the resulting column without sampling.


In [ ]:
# Tier each transaction by amount.
# The `-> str` return-type hint is what lets Spark skip the sampling step.
def spend_tier(amount: float) -> str:
    if amount > 50000:
        return "HIGH"
    if amount > 10000:
        return "STANDARD"
    if amount > 1000:
        return "LOW"
    return "MICRO"

# `apply` on a Series — element-wise
psdf["tier"] = psdf["amount"].apply(spend_tier)

# Still lazy — the head call in the next section is the action
print("tier column added (no job run)")


## Actions — eager, trigger a Spark job

Each line in this section *fires a job*. The plan built up by the transformations above runs all the way through Catalyst, materializes, and returns either pandas-side data or a write side effect.

The five families:

- **Display / preview** — `head`, `tail`, `print(df)`
- **Driver materialization** — `to_pandas`, `.values`, `.iloc[i]`
- **Frame-wide reductions** — `count`, `sum`, `mean`, `describe`, `len(df)`
- **Writes** — `to_csv`, `to_parquet`, `to_table`
- **Iteration** — `iterrows`, `itertuples` (avoid on large data)


### Display and preview

`head(n)` is the most common action — it's how you sanity-check a pipeline mid-build. Each call runs the full plan up to that point and returns a small pandas DataFrame.


In [ ]:
# Each of these triggers a Spark job
print("--- psdf.head(3) — fires a job, returns a pandas DataFrame ---")
print(psdf.head(3))

print()
print("--- print(psdf) — implicitly calls head, also a job ---")
print(psdf[["transaction_id", "amount", "tier"]])


### Driver materialization — `to_pandas` and friends

These pull the entire frame to the driver. Same trap as `df.toPandas()` and `rdd.collect()` from earlier notebooks — fine on aggregated results, catastrophic on raw frames. Always reduce first.


In [ ]:
# Safe: aggregate first, then collect the small result
small_pdf = (
    psdf[psdf["status"] == "APPROVED"]
        .groupby("merchant_category")
        .size()
        .to_pandas()
)
print("type :", type(small_pdf).__module__)
print(small_pdf)

# DANGEROUS on a large frame — do not uncomment in real code
# everything_pdf = psdf.to_pandas()    # would pull all rows to the driver


### Frame-wide reductions

Calling `.sum()`, `.mean()`, `.count()`, or `.describe()` on a whole frame runs an aggregation Spark job and returns a pandas Series (or scalar). `len(psdf)` is the equivalent of `df.count()` — also a job.


In [ ]:
print("len(psdf)     :", len(psdf))           # row count — Spark job
print("amount sum    :", float(psdf["amount"].sum()))
print("amount mean   :", float(psdf["amount"].mean()))
print()
print(psdf[["amount"]].describe())            # count / mean / std / min / max — Spark job


### Writes

`to_csv`, `to_parquet`, and `to_table` all run jobs that write distributed output through Spark's standard writers. They accept the usual `mode=` and `partitionBy=` arguments.


In [ ]:
import shutil, os
out_path = "_scratch_11/ps_tier_summary"
shutil.rmtree(out_path, ignore_errors=True)

(
    psdf.groupby("tier")
        .agg(n=("transaction_id", "count"), total=("amount", "sum"))
        .reset_index()
        .to_parquet(out_path, mode="overwrite")
)

print("wrote:", sorted(os.listdir(out_path))[:3], "...")


### Iteration — almost always wrong

`for row in psdf.iterrows()` pulls every row to the driver before iterating. This collapses the whole point of using Spark. Reach for vectorized ops (column expressions, `apply` with a type hint, or `mapInPandas` on the Spark DataFrame) instead.


## The Spark bridge — `to_spark()` and `pandas_api()`

The two ends of the wrapper are cheap to flip between. Neither moves data — they swap whether the index is exposed pandas-style or hidden Spark-style.

| Direction | Method | Cost |
|---|---|---|
| Spark DF → Pandas-on-Spark | `sdf.pandas_api()` | Cheap — just a wrapper |
| Pandas-on-Spark → Spark DF | `psdf.to_spark()` | Cheap — unwraps the wrapper |
| Spark DF → pandas | `sdf.toPandas()` | **Expensive** — driver collect |
| Pandas-on-Spark → pandas | `psdf.to_pandas()` | **Expensive** — driver collect |
| pandas → Pandas-on-Spark | `ps.from_pandas(pdf)` | Reads the pandas frame through Spark |
| pandas → Spark DF | `spark.createDataFrame(pdf)` | Reads the pandas frame through Spark |

The methods with lowercase singular `pandas` in the name — `toPandas`, `to_pandas` — move data to the driver. Treat them like `collect()`.


In [ ]:
# Round trip: pandas-on-Spark → Spark → pandas-on-Spark
spark_view = psdf.to_spark()
print("to_spark    :", type(spark_view).__module__)

psdf_again = spark_view.pandas_api()
print("pandas_api  :", type(psdf_again).__module__)

# Useful pattern: flip to Spark to use a Spark-only feature (e.g. a SQL view, broadcast join hint),
# then flip back to continue in pandas-on-Spark.
spark_view.createOrReplaceTempView("v_txns")
top_cats = spark.sql("""
    SELECT merchant_category, COUNT(*) AS n
    FROM v_txns
    WHERE status = 'APPROVED'
    GROUP BY merchant_category
    ORDER BY n DESC
    LIMIT 5
""").pandas_api()

print(top_cats)


## Default index types — the most-tested gotcha

Pandas DataFrames have a built-in row index. Spark DataFrames do not — rows have no inherent order. Pandas-on-Spark has to *fabricate* an index when one isn't provided, and the strategy it picks has both performance and correctness implications.

Three strategies, set via the `compute.default_index_type` option:

| Index type | How it's computed | Cost | Order guarantee |
|---|---|---|---|
| **`sequence`** | Single executor counts `0, 1, 2, ...` | Triggers a non-distributed step | Sequential — matches pandas |
| **`distributed-sequence`** (current default) | Computed in parallel using partition offsets | Slight overhead | Sequential — same numbers as pandas |
| **`distributed`** | Monotonically increasing IDs (`monotonically_increasing_id`) | Cheapest — fully parallel | **Not sequential** — large gaps allowed |

The trap is `distributed`: you get `0, 1, 8589934592, 8589934593, ...`. Code that depends on the exact index values (`loc`-based lookups, joins on index, `.iloc[5]` semantics on an unsorted axis) will silently break.

`sequence` is safe but slow on large data. `distributed-sequence` is the modern default and the right answer for most cases.


In [ ]:
print("Current default index type :", ps.get_option("compute.default_index_type"))

for idx_type in ["sequence", "distributed-sequence", "distributed"]:
    ps.set_option("compute.default_index_type", idx_type)
    psdf_demo = card_transactions_sdf.pandas_api()
    head_idx = list(psdf_demo.head(5).index)
    print(f"  {idx_type:22s} : head indices = {head_idx}")

# Reset to the safe modern default
ps.set_option("compute.default_index_type", "distributed-sequence")


## Gotchas — where the pandas API ends

The surface is *close* to pandas, but not identical. The exam loves these differences.

| Behaviour | pandas | Pandas API on Spark |
|---|---|---|
| Row order | Stable, follows insertion | **Not guaranteed** unless you `sort_index()` / `sort_values()` |
| Lazy vs eager | Eager — every op runs immediately | **Lazy** — only actions trigger execution |
| `inplace=True` | Modifies in place | Often ignored — Spark DataFrames are immutable |
| Iteration (`iterrows`) | Cheap | Pulls all rows to driver — avoid |
| `df.apply(func, axis=1)` without type hint | Runs immediately | Samples rows to infer schema — slow and brittle |
| `df.loc[label]` with missing label | `KeyError` | May silently return empty — index not fully enforced |
| Ops across two frames | Always allowed | **Off by default** — must set `compute.ops_on_diff_frames` |
| Some methods | All of them | A handful raise `PandasNotImplementedError` |

The two settings worth knowing:

- **`compute.ops_on_diff_frames`** — when `False` (default), operations that combine two pandas-on-Spark frames raise an error, because the row-alignment semantics of pandas require both frames to share an index, and Spark cannot guarantee that without a shuffle. Set to `True` only when you've reasoned about the cost.
- **`compute.default_index_type`** — covered above.

**Performance rule of thumb.** If a pandas-on-Spark op maps to a single Spark DataFrame operation (`filter`, `groupBy`, `join`, `withColumn`), it's fast. If it forces a per-row Python callback without a type hint, or materializes the whole frame on the driver, it's slow — at that scale you're better off in the native DataFrame API.


In [ ]:
# ops_on_diff_frames — disabled by default
print("ops_on_diff_frames :", ps.get_option("compute.ops_on_diff_frames"))

# Build two frames and try to combine columns from each
a = psdf[["transaction_id", "amount"]].head(5).reset_index(drop=True)
b = psdf[["status"]].head(5).reset_index(drop=True)

try:
    a["status"] = b["status"]
except Exception as e:
    print("expected failure :", type(e).__name__)

# Enable only when you've reasoned about the shuffle cost
ps.set_option("compute.ops_on_diff_frames", True)
a["status"] = b["status"]
print(a)
ps.set_option("compute.ops_on_diff_frames", False)


In [ ]:
spark.stop()
